# 05 — Explainability Analysis

Grad-CAM heatmaps, eight-region facial activation scoring, TP/TN/FP/FN case comparison.

**Requires:** Trained model (.h5), test dataset.

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import dlib
from tqdm import tqdm

## Setup

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

PREDICTOR_PATH = "/content/drive/MyDrive/FF_Data/shape_predictor_68_face_landmarks.dat"
face_detector = dlib.get_frontal_face_detector()
shape_predictor = dlib.shape_predictor(PREDICTOR_PATH)

def extract_landmarks(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    faces = face_detector(gray)
    if not faces:
        return None
    shape = shape_predictor(gray, faces[0])
    return [(p.x, p.y) for p in shape.parts()]

def denormalize(img):
    x = img.astype(np.float32).copy()
    for i in range(3):
        x[..., i] = (x[..., i] * STD[i]) + MEAN[i]
    return np.clip(x * 255, 0, 255).astype(np.uint8)

## Grad-CAM Engine

Extracts the last conv layer from EfficientNet-B4 inside the TimeDistributed wrapper, computes gradients of the predicted class score w.r.t. that layer, and produces a heatmap.

No `tf.cast` inside the model graph (breaks Keras 3). Casting is done in eager mode after inference.

In [ ]:
def get_backbone(video_model):
    for l in video_model.layers:
        if isinstance(l, tf.keras.layers.TimeDistributed) and isinstance(l.layer, tf.keras.Model):
            return l.layer
    raise ValueError("TimeDistributed backbone not found.")

def get_last_conv_name(backbone):
    for layer in reversed(backbone.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D found.")

def find_td_layer(video_model, layer_cls):
    for l in video_model.layers:
        if isinstance(l, tf.keras.layers.TimeDistributed) and isinstance(l.layer, layer_cls):
            return l
    raise ValueError(f"TimeDistributed({layer_cls.__name__}) not found")

def build_frame_grad_model(video_model):
    """Frame-level model returning (conv_output, prediction)."""
    backbone = get_backbone(video_model)
    last_conv = get_last_conv_name(backbone)

    conv_and_end = tf.keras.Model(
        inputs=backbone.input,
        outputs=[backbone.get_layer(last_conv).output, backbone.output],
        name="backbone_conv_end")

    td_gap  = find_td_layer(video_model, tf.keras.layers.GlobalAveragePooling2D)
    td_drop = find_td_layer(video_model, tf.keras.layers.Dropout)
    td_dense = find_td_layer(video_model, tf.keras.layers.Dense)

    inp = tf.keras.Input(shape=(224, 224, 3))
    conv_out, feats = conv_and_end(inp)
    x = td_gap.layer(feats)
    x = td_drop.layer(x, training=False)
    prob = td_dense.layer(x)

    return tf.keras.Model(inp, [conv_out, prob], name="frame_grad_model")

In [ ]:
def class_score(prob, target="auto"):
    p = tf.clip_by_value(tf.squeeze(prob, axis=-1), 1e-6, 1.0 - 1e-6)
    if target == "real":
        return tf.math.log(p)
    elif target == "fake":
        return tf.math.log(1.0 - p)
    return tf.where(p >= 0.5, tf.math.log(p), tf.math.log(1.0 - p))

def compute_gradcam(video_model, frame_norm, target="auto"):
    """Grad-CAM on one frame. Returns (overlay, heatmap, original)."""
    grad_model = build_frame_grad_model(video_model)
    inp = tf.expand_dims(tf.cast(frame_norm, tf.float32), axis=0)

    with tf.GradientTape() as tape:
        conv, prob = grad_model(inp, training=False)
        loss = class_score(prob, target=target)

    grads = tape.gradient(loss, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2))

    # eager -> numpy, cast to float32 for mixed precision safety
    conv_np = conv[0].numpy().astype(np.float32)
    weights_np = weights[0].numpy().astype(np.float32)

    cam = np.sum(conv_np * weights_np, axis=-1)
    cam = np.maximum(cam, 0)
    if cam.max() > 0:
        cam /= cam.max()
    cam = cv2.resize(cam, (224, 224))

    frame_rgb = denormalize(frame_norm)
    colormap = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
    colormap = cv2.cvtColor(colormap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(frame_rgb, 0.70, colormap, 0.30, 0)

    return overlay, cam, frame_rgb

## Regional Activation Analysis

Eight facial regions from dlib 68-point landmarks. Eye labels remapped to image-left/right (dlib uses person-left/right). Masks dilated 2px so edge activations aren't lost.

In [ ]:
REGION_IDX = {
    "jaw":           list(range(0, 17)),
    "right_eyebrow": list(range(17, 22)),
    "left_eyebrow":  list(range(22, 27)),
    "nose":          list(range(27, 36)),
    "right_eye":     list(range(36, 42)),
    "left_eye":      list(range(42, 48)),
    "outer_mouth":   list(range(48, 60)),
    "inner_mouth":   list(range(60, 68)),
}

def _centroid(pts):
    pts = np.asarray(pts, np.float32)
    return float(pts[:, 0].mean()), float(pts[:, 1].mean())

def region_map_by_image_side(landmarks):
    """Remap dlib L/R to image L/R by x-coordinate."""
    cx_r_eye, _ = _centroid([landmarks[i] for i in REGION_IDX["right_eye"]])
    cx_l_eye, _ = _centroid([landmarks[i] for i in REGION_IDX["left_eye"]])

    if cx_r_eye < cx_l_eye:
        eye_left_idx, eye_right_idx = REGION_IDX["right_eye"], REGION_IDX["left_eye"]
    else:
        eye_left_idx, eye_right_idx = REGION_IDX["left_eye"], REGION_IDX["right_eye"]

    cx_r_br, _ = _centroid([landmarks[i] for i in REGION_IDX["right_eyebrow"]])
    cx_l_br, _ = _centroid([landmarks[i] for i in REGION_IDX["left_eyebrow"]])

    if cx_r_br < cx_l_br:
        brow_left_idx, brow_right_idx = REGION_IDX["right_eyebrow"], REGION_IDX["left_eyebrow"]
    else:
        brow_left_idx, brow_right_idx = REGION_IDX["left_eyebrow"], REGION_IDX["right_eyebrow"]

    return {
        "eye_left": eye_left_idx, "eye_right": eye_right_idx,
        "brow_left": brow_left_idx, "brow_right": brow_right_idx,
        "nose": REGION_IDX["nose"],
        "outer_mouth": REGION_IDX["outer_mouth"],
        "inner_mouth": REGION_IDX["inner_mouth"],
        "jaw": REGION_IDX["jaw"],
    }

def region_scores(heatmap, landmarks, dilate_px=2):
    """Per-region mean activation (0-100 scale)."""
    regions = region_map_by_image_side(landmarks)
    scores = {}

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (2*dilate_px+1, 2*dilate_px+1)) if dilate_px > 0 else None

    for name, idxs in regions.items():
        poly = np.array([landmarks[i] for i in idxs], np.int32)
        mask = np.zeros_like(heatmap, dtype=np.uint8)
        cv2.fillPoly(mask, [poly], 1)
        if kernel is not None:
            mask = cv2.dilate(mask, kernel, iterations=1)
        m = mask.astype(bool)
        scores[name] = float(heatmap[m].mean() * 100) if m.any() else 0.0

    return scores

## Case Collection

In [ ]:
def collect_cases(model, dataset, threshold=0.5):
    cases = {"TP": [], "TN": [], "FP": [], "FN": []}

    for batch_idx in tqdm(range(len(dataset)), desc="Collecting"):
        X, y = dataset[batch_idx]
        preds = model.predict(X, verbose=0)

        for i in range(X.shape[0]):
            if i % 4 >= 2:  # skip augmented copies
                continue

            yt = int(y[i, 0, 0])
            yp = float(np.mean(preds[i]))
            yc = 1 if yp >= threshold else 0

            if   yt == 0 and yc == 0: cases["TP"].append(X[i])
            elif yt == 1 and yc == 1: cases["TN"].append(X[i])
            elif yt == 1 and yc == 0: cases["FP"].append(X[i])
            elif yt == 0 and yc == 1: cases["FN"].append(X[i])

    for k, v in cases.items():
        print(f"  {k}: {len(v)}")
    return cases

cases = collect_cases(model, test_dataset)

## Average Activation per Case

In [ ]:
def compute_average_regions(model, cases, frame_indices=[0, 4, 7],
                           max_samples=None):
    result = {}

    for case_type in ["TP", "TN", "FP", "FN"]:
        sequences = cases[case_type]
        if max_samples and len(sequences) > max_samples:
            sequences = random.sample(sequences, max_samples)

        all_scores = []
        for seq in tqdm(sequences, desc=case_type):
            try:
                heats = []
                for fi in frame_indices:
                    fi = min(fi, len(seq) - 1)
                    _, h, _ = compute_gradcam(model, seq[fi])
                    heats.append(h)
                avg_heat = np.mean(heats, axis=0)

                mid_rgb = denormalize(seq[len(seq) // 2])
                lm = extract_landmarks(mid_rgb)
                if lm is not None:
                    all_scores.append(region_scores(avg_heat, lm))
            except Exception:
                continue

        if all_scores:
            all_regions = set()
            for s in all_scores:
                all_regions.update(s.keys())
            result[case_type] = {
                r: np.mean([s[r] for s in all_scores if r in s])
                for r in all_regions}
        else:
            result[case_type] = {}

    return result

import random
case_avg = compute_average_regions(model, cases)

## Gridmap: Region x Case

In [ ]:
def plot_gridmap(case_avg):
    case_order = ["TN", "TP", "FP", "FN"]
    all_regions = sorted(set(r for sc in case_avg.values() for r in sc))

    matrix = np.zeros((len(case_order), len(all_regions)))
    for i, c in enumerate(case_order):
        for j, r in enumerate(all_regions):
            matrix[i, j] = case_avg.get(c, {}).get(r, 0)

    labels = [r.replace("_", " ").title() for r in all_regions]

    plt.figure(figsize=(12, 5))
    sns.heatmap(matrix, annot=True, fmt=".0f", cmap="RdYlGn_r",
                vmin=0, vmax=100, xticklabels=labels,
                yticklabels=case_order,
                linewidths=1, linecolor="white")
    plt.title("Regional Activation: Cases vs Facial Regions")
    plt.xlabel("Facial Region"); plt.ylabel("Case")
    plt.axhline(y=2, color="blue", linewidth=2)
    plt.tight_layout(); plt.show()

plot_gridmap(case_avg)

## Single-Sample Grad-CAM

In [ ]:
def show_gradcam_pair(model, real_seq, fake_seq, frame_idx=7, threshold=0.5):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    for ax, seq, label in zip(axes, [real_seq, fake_seq], ["Real", "Fake"]):
        fi = min(frame_idx, len(seq) - 1)
        overlay, _, _ = compute_gradcam(model, seq[fi])
        score = float(model.predict(np.expand_dims(seq, 0), verbose=0).mean())
        pred_label = "REAL" if score >= threshold else "FAKE"

        ax.imshow(overlay)
        ax.set_title(f"{label} | pred={score:.2f} ({pred_label})")
        ax.axis("off")

    plt.tight_layout(); plt.show()

# example: first batch, first real and fake
X, y = test_dataset[0]
r_idx = np.where(y[:, 0, 0] > 0.5)[0][0]
f_idx = np.where(y[:, 0, 0] <= 0.5)[0][0]
show_gradcam_pair(model, X[r_idx], X[f_idx])